In [1]:
%pip install PyWavelets antropy --quiet

Note: you may need to restart the kernel to use updated packages.


# Первый пайплайн для дыхания, SpO2 и PPG

In [2]:
import numpy as np
import pandas as pd
import mne
from scipy import signal
from scipy.signal import find_peaks
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')
import antropy as ant

In [3]:
# ------------------------------------------------------------
# 1. Функции загрузки, предобработки и извлечения признаков
# ------------------------------------------------------------

def load_edf_signals(edf_path, channels):
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    raw.pick_channels(channels)
    sfreq = raw.info['sfreq']
    signals = {ch: raw[ch][0].flatten() for ch in channels}
    return raw, signals, sfreq

def preprocess_signal(data, sfreq, lowcut=0.1, highcut=10, order=4):
    nyquist = 0.5 * sfreq
    low = lowcut / nyquist
    high = highcut / nyquist
    if low <= 0 or low >= high:
        b, a = signal.butter(order, high, btype='low')
    else:
        b, a = signal.butter(order, [low, high], btype='band')
    return signal.filtfilt(b, a, data)

def resample_signal(data, orig_sfreq, target_sfreq):
    if orig_sfreq == target_sfreq:
        return data
    num_samples = int(len(data) * target_sfreq / orig_sfreq)
    return signal.resample(data, num_samples)

def detect_resp_rate(resp_signal, sfreq):
    min_distance = int(2 * sfreq)
    peaks, _ = find_peaks(resp_signal, distance=min_distance, prominence=np.std(resp_signal)*0.2)
    if len(peaks) < 2:
        return np.nan
    intervals = np.diff(peaks) / sfreq
    return 60 / np.mean(intervals)

def detect_heart_rate(ppg_signal, sfreq):
    low = 0.5 / (sfreq/2)
    high = 5.0 / (sfreq/2)
    b, a = signal.butter(2, [low, high], btype='band')
    filtered_ppg = signal.filtfilt(b, a, ppg_signal)
    min_distance = int(0.5 * sfreq)
    peaks, _ = find_peaks(filtered_ppg, distance=min_distance, prominence=np.std(filtered_ppg)*0.5)
    if len(peaks) < 2:
        return np.nan, np.nan, np.nan, np.nan
    rr_intervals = np.diff(peaks) / sfreq
    heart_rate = 60 / np.mean(rr_intervals)
    rmssd = np.sqrt(np.mean(np.diff(rr_intervals)**2))
    sdnn = np.std(rr_intervals)
    nn50 = np.sum(np.abs(np.diff(rr_intervals)) > 0.05)
    pnn50 = nn50 / len(rr_intervals) * 100
    return heart_rate, rmssd, sdnn, pnn50

def extract_features_from_window(window_signals, sfreq, channel_names):
    features = {}
    for ch in channel_names:
        data = window_signals[ch]
        if ch.startswith('RESP Breath'):
            features[f'{ch}_mean'] = np.mean(data)
            features[f'{ch}_std'] = np.std(data)
            features[f'{ch}_min'] = np.min(data)
            features[f'{ch}_max'] = np.max(data)
            features[f'{ch}_p10'] = np.percentile(data, 10)
            features[f'{ch}_p90'] = np.percentile(data, 90)
            features[f'{ch}_slope'] = (data[-1] - data[0]) / len(data)
            resp_rate = detect_resp_rate(data, sfreq)
            features[f'{ch}_resp_rate'] = resp_rate if not np.isnan(resp_rate) else 0
            entropy = ant.spectral_entropy(data, sfreq, method='welch')
            features[f'{ch}_entropy'] = entropy
        elif ch == 'SAO2 SpO2':
            features[f'{ch}_mean'] = np.mean(data)
            features[f'{ch}_std'] = np.std(data)
            features[f'{ch}_min'] = np.min(data)
            features[f'{ch}_desat_count_3pct'] = np.sum(np.diff(data) <= -3)
        elif ch == 'PPG':
            features[f'{ch}_mean'] = np.mean(data)
            features[f'{ch}_std'] = np.std(data)
            hr, rmssd, sdnn, pnn50 = detect_heart_rate(data, sfreq)
            features[f'{ch}_heart_rate'] = hr if not np.isnan(hr) else 0
            features[f'{ch}_rmssd'] = rmssd if not np.isnan(rmssd) else 0
            features[f'{ch}_sdnn'] = sdnn if not np.isnan(sdnn) else 0
            features[f'{ch}_pnn50'] = pnn50 if not np.isnan(pnn50) else 0
    return features

def extract_epochs_dataset(raw, signals, sfreq, epoch_duration=30, 
                           channel_names=None, stage_mapping=None):
    """
    Извлекает эпохи и метки сна из аннотаций.
    stage_mapping: dict {строка_аннотации: числовая_метка}
    """
    if stage_mapping is None:
        stage_mapping = {
            'Sleep stage W(eventUnknown)': 0,
            'Sleep stage 1(eventUnknown)': 1,
            'Sleep stage 2(eventUnknown)': 2,
            'Sleep stage 3(eventUnknown)': 3,
            'Sleep stage R(eventUnknown)': 4,
        }
    
    if channel_names is None:
        channel_names = list(signals.keys())
    
    n_samples = len(signals[channel_names[0]])
    duration_sec = n_samples / sfreq
    n_epochs = int(duration_sec // epoch_duration)
    
    y_epochs = [None] * n_epochs
    
    for ann in raw.annotations:
        desc = ann['description']
        if desc not in stage_mapping:
            continue
        stage = stage_mapping[desc]
        onset = ann['onset']
        duration = ann['duration']
        start_epoch = int(onset // epoch_duration)
        end_epoch = int((onset + duration) // epoch_duration) + 1
        start_epoch = max(0, start_epoch)
        end_epoch = min(n_epochs, end_epoch)
        for i in range(start_epoch, end_epoch):
            if y_epochs[i] is None:
                y_epochs[i] = stage
    
    valid_epochs = [i for i, label in enumerate(y_epochs) if label is not None]
    if len(valid_epochs) == 0:
        raise ValueError("Нет эпох с аннотациями фаз сна.")
    
    X_list = []
    y_list = []
    for i in tqdm(valid_epochs, desc=f"Извлечение признаков ({os.path.basename(raw.filenames[0])})", leave=False):
        start_sample = int(i * epoch_duration * sfreq)
        end_sample = int((i+1) * epoch_duration * sfreq)
        window_signals = {ch: signals[ch][start_sample:end_sample] for ch in channel_names}
        feat_dict = extract_features_from_window(window_signals, sfreq, channel_names)
        X_list.append(feat_dict)
        y_list.append(y_epochs[i])
    
    X_df = pd.DataFrame(X_list).fillna(0)
    return X_df, np.array(y_list)

In [4]:
class IncrementalScaler:
    """
    Вычисляет среднее и стандартное отклонение потоковым способом (Welford).
    Позволяет обновлять статистику по батчам.
    """
    def __init__(self):
        self.n = 0
        self.mean = None
        self.M2 = None  # сумма квадратов разностей
        self.var = None
        self.std = None
    
    def partial_fit(self, X):
        """X: numpy array (n_samples, n_features)"""
        if X.ndim == 1:
            X = X.reshape(1, -1)
        if self.mean is None:
            self.mean = np.zeros(X.shape[1])
            self.M2 = np.zeros(X.shape[1])
        for x in X:
            self.n += 1
            delta = x - self.mean
            self.mean += delta / self.n
            delta2 = x - self.mean
            self.M2 += delta * delta2
        if self.n > 1:
            self.var = self.M2 / (self.n - 1)
            self.std = np.sqrt(self.var)
        else:
            self.var = np.zeros_like(self.mean)
            self.std = np.ones_like(self.mean)
        return self
    
    def transform(self, X):
        """Применяет масштабирование (X - mean) / std"""
        if self.std is None:
            raise ValueError("Сначала вызовите partial_fit")
        if X.ndim == 1:
            X = X.reshape(1, -1)
        return (X - self.mean) / (self.std + 1e-8)

In [5]:
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd
import os

def train_incremental(edf_files, channels, stage_mapping=None, 
                      target_sfreq=25, epoch_duration=30,
                      first_file_for_scaler=None):
    """
    Обучает SGDClassifier инкрементально по файлам.
    
    Параметры:
        edf_files: список путей к файлам
        channels: список каналов
        first_file_for_scaler: если задан, масштабируется по статистикам из этого файла.
                              Иначе используется IncrementalScaler, обучаемый на всех данных.
    """
    if stage_mapping is None:
        stage_mapping = {
            'Sleep stage W(eventUnknown)': 0,
            'Sleep stage 1(eventUnknown)': 1,
            'Sleep stage 2(eventUnknown)': 2,
            'Sleep stage 3(eventUnknown)': 3,
            'Sleep stage R(eventUnknown)': 4,
        }
    n_classes = len(set(stage_mapping.values()))
    
    # Определяем классы (0,1,2,3,4)
    all_classes = np.arange(n_classes)
    
    # 1. Если нужен фиксированный масштабатор, обучим его на первом файле
    if first_file_for_scaler:
        print(f"Обучение масштабатора на {first_file_for_scaler}...")
        raw, signals, sfreq = load_edf_signals(first_file_for_scaler, channels)
        proc = preprocess_all_signals(signals, sfreq, target_sfreq, channels)
        X_scaler, _ = extract_epochs_dataset(
            raw, proc, target_sfreq, epoch_duration, channels, stage_mapping
        )
        scaler = StandardScaler()
        scaler.fit(X_scaler)
        print("Масштабатор готов")
    else:
        scaler = IncrementalScaler()
        scaler_ready = False
    
    # 2. Создаём модель
    model = SGDClassifier(loss='log_loss',  # логистическая регрессия (вероятности)
                          penalty='l2',
                          alpha=0.0001,
                          max_iter=1,       # один проход по батчу
                          tol=None,
                          warm_start=True,  # позволяет продолжать обучение
                          random_state=42,
                          n_jobs=-1)
    
    # 3. Инкрементальное обучение по файлам
    total_epochs = 0
    filenames_processed = []
    for idx, edf_path in enumerate(edf_files):
        print(f"\nОбработка файла {idx+1}/{len(edf_files)}: {os.path.basename(edf_path)}")
        try:
            # Загрузка и предобработка
            raw, signals, sfreq = load_edf_signals(edf_path, channels)
            processed = preprocess_all_signals(signals, sfreq, target_sfreq, channels)
            
            # Извлечение признаков и меток (это уже делает X, y)
            X_file, y_file = extract_epochs_dataset(
                raw, processed, target_sfreq, epoch_duration, channels, stage_mapping
            )
            if len(X_file) == 0:
                print("  -> Нет эпох с аннотациями, пропускаем")
                continue
            
            # Масштабирование признаков
            if first_file_for_scaler:
                X_scaled = scaler.transform(X_file)
            else:
                if not scaler_ready:
                    # Первый файл: partial_fit scaling, затем тренировка
                    scaler.partial_fit(X_file.values)
                    scaler_ready = True
                X_scaled = scaler.transform(X_file.values)
            
            # Инкрементальное обучение модели
            if idx == 0:
                # При первом вызове нужно указать все классы
                model.partial_fit(X_scaled, y_file, classes=all_classes)
            else:
                model.partial_fit(X_scaled, y_file)
            
            total_epochs += len(y_file)
            filenames_processed.append(os.path.basename(edf_path))
            print(f"  -> Добавлено {len(y_file)} эпох, всего эпох: {total_epochs}")
            
        except Exception as e:
            print(f"  -> Ошибка: {e}, пропускаем файл")
            continue
    
    print(f"\n✅ Обучение завершено. Обработано файлов: {len(filenames_processed)}, всего эпох: {total_epochs}")
    return model, scaler, filenames_processed

def preprocess_all_signals(signals, orig_sfreq, target_sfreq, channels):
    """Вспомогательная функция: фильтрация и ресемплинг."""
    processed = {}
    for ch, data in signals.items():
        if ch.startswith('RESP Breath'):
            filtered = preprocess_signal(data, orig_sfreq, lowcut=0.05, highcut=2)
        elif ch == 'SAO2 SpO2':
            filtered = preprocess_signal(data, orig_sfreq, lowcut=0, highcut=0.5)
        elif ch == 'PPG':
            filtered = preprocess_signal(data, orig_sfreq, lowcut=0.5, highcut=8)
        else:
            filtered = data
        processed[ch] = resample_signal(filtered, orig_sfreq, target_sfreq)
    return processed

In [6]:
# Список всех EDF файлов
almazov_data = '../dataset/edf/'

# Список всех ваших EDF файлов
all_files = [
    almazov_data + '111.edf',
    almazov_data + '196.edf',
    almazov_data + '222.edf',
    almazov_data + '233.edf',
    # almazov_data + '303.edf',
    # almazov_data + '304.edf',
    # almazov_data + '308.edf',
]

# Каналы, которые вы используете
channels = ['RESP Breath-0', 'SAO2 SpO2', 'PPG', 'RESP Breath-1']

# Другие параметры
target_sfreq = 25
epoch_duration = 30

In [7]:
# Обучаем инкрементально
model, scaler, processed_files = train_incremental(
    edf_files=all_files,
    channels=channels,
    stage_mapping=None,   # использует стандартные названия
    target_sfreq=target_sfreq,
    epoch_duration=epoch_duration,
    first_file_for_scaler=all_files[0]   # масштабируем по статистикам первого файла
)

Обучение масштабатора на ../dataset/edf/111.edf...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


Масштабатор готов

Обработка файла 1/4: 111.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


  -> Добавлено 1550 эпох, всего эпох: 1550

Обработка файла 2/4: 196.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


  -> Добавлено 1570 эпох, всего эпох: 3120

Обработка файла 3/4: 222.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


  -> Добавлено 1803 эпох, всего эпох: 4923

Обработка файла 4/4: 233.edf
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).


  -> Добавлено 1320 эпох, всего эпох: 6243

✅ Обучение завершено. Обработано файлов: 4, всего эпох: 6243


In [18]:
def evaluate_on_file(edf_path, model, scaler, channels, target_sfreq=25):
    """Предсказывает фазы для одного файла и печатает отчёт (если есть истинные метки)."""
    raw, signals, sfreq = load_edf_signals(edf_path, channels)
    processed = preprocess_all_signals(signals, sfreq, target_sfreq, channels)
    X_test, y_test = extract_epochs_dataset(raw, processed, sfreq)
    X_scaled = scaler.transform(X_test)
    y_pred = model.predict(X_scaled)
    # Дополнительно: вероятности, если нужны для ROC
    y_proba = model.predict_proba(X_scaled) if hasattr(model, 'predict_proba') else None
    return X_test, X_scaled, y_test, y_pred, y_proba

In [15]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, precision_recall_fscore_support, roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

def plot_detailed_report(y_true, y_pred, model=None, X_test=None, feature_names=None, class_names=None):
    """
    Строит развёрнутые графики для оценки классификации фаз сна.
    
    Параметры:
    ----------
    y_true : array-like
        Истинные метки классов.
    y_pred : array-like
        Предсказанные метки.
    model : объект с методом predict_proba (опционально)
        Если передан, строятся ROC-кривые.
    X_test : DataFrame или array (опционально)
        Признаки тестового набора (нужны для важности признаков).
    feature_names : list (опционально)
        Имена признаков.
    class_names : list (опционально)
        Названия классов по порядку (по умолчанию ['W', 'N1', 'N2', 'N3', 'REM']).
    """
    if class_names is None:
        class_names = ['W', 'N1', 'N2', 'N3', 'REM']
    n_classes = len(class_names)
    
    # 1. Матрица ошибок
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, 
                cmap='Blues', ax=axes[0])
    axes[0].set_title('Confusion Matrix (absolute)')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    
    sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', ax=axes[1])
    axes[1].set_title('Confusion Matrix (normalized by row)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    plt.tight_layout()
    plt.show()
    
    # 2. Precision, Recall, F1 по классам (столбчатая диаграмма)
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None)
    
    x = np.arange(n_classes)
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(10, 6))
    bars1 = ax.bar(x - width, precision, width, label='Precision', color='skyblue')
    bars2 = ax.bar(x, recall, width, label='Recall', color='lightgreen')
    bars3 = ax.bar(x + width, f1, width, label='F1-score', color='salmon')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Classification metrics per sleep stage')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    
    # Подписи значений на столбцах
    for i, (p, r, f) in enumerate(zip(precision, recall, f1)):
        ax.text(i - width, p + 0.02, f'{p:.2f}', ha='center', va='bottom', fontsize=9)
        ax.text(i, r + 0.02, f'{r:.2f}', ha='center', va='bottom', fontsize=9)
        ax.text(i + width, f + 0.02, f'{f:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()
    
    # 3. Сравнение распределений истинных и предсказанных классов
    true_counts = np.bincount(y_true, minlength=n_classes)
    pred_counts = np.bincount(y_pred, minlength=n_classes)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(n_classes)
    ax.bar(x - 0.2, true_counts, width=0.4, label='True', color='steelblue')
    ax.bar(x + 0.2, pred_counts, width=0.4, label='Predicted', color='darkorange')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names)
    ax.set_ylabel('Number of epochs')
    ax.set_title('Distribution of sleep stages: True vs Predicted')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # 4. ROC-кривые (если модель поддерживает predict_proba)
    if model is not None and hasattr(model, 'predict_proba'):
        try:
            y_score = model.predict_proba(X_test)
            y_true_bin = label_binarize(y_true, classes=np.arange(n_classes))
            fpr = dict()
            tpr = dict()
            roc_auc = dict()
            for i in range(n_classes):
                fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_score[:, i])
                roc_auc[i] = auc(fpr[i], tpr[i])
            
            plt.figure(figsize=(8, 6))
            colors = ['blue', 'orange', 'green', 'red', 'purple']
            for i, color in zip(range(n_classes), colors):
                plt.plot(fpr[i], tpr[i], color=color, lw=2,
                         label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')
            plt.plot([0, 1], [0, 1], 'k--', lw=1)
            plt.xlim([0.0, 1.0])
            plt.ylim([0.0, 1.05])
            plt.xlabel('False Positive Rate')
            plt.ylabel('True Positive Rate')
            plt.title('ROC curves (One-vs-Rest)')
            plt.legend(loc='lower right')
            plt.grid(alpha=0.3)
            plt.show()
        except Exception as e:
            print(f"Не удалось построить ROC-кривые: {e}")
    else:
        print("Модель не поддерживает predict_proba или не передана. ROC-кривые пропущены.")
    
    # 5. Важность признаков (для моделей с coef_ или feature_importances_)
    if X_test is not None and feature_names is not None:
        if hasattr(model, 'coef_'):
            # Линейные модели (SGDClassifier, LogisticRegression) - берём средний абсолютный коэффициент по классам
            importances = np.abs(model.coef_).mean(axis=0)
            title = 'Mean absolute coefficients (feature importance)'
        elif hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            title = 'Feature importances'
        else:
            importances = None
        
        if importances is not None:
            # Берём топ-20 признаков (или все, если меньше 20)
            n_top = min(20, len(importances))
            indices = np.argsort(importances)[::-1][:n_top]
            plt.figure(figsize=(10, 6))
            plt.barh(range(n_top), importances[indices], align='center')
            plt.yticks(range(n_top), [feature_names[i] for i in indices])
            plt.xlabel('Importance')
            plt.title(title)
            plt.gca().invert_yaxis()
            plt.tight_layout()
            plt.show()
        else:
            print("Модель не предоставляет важность признаков (нет coef_ или feature_importances_).")
    else:
        print("X_test или feature_names не переданы, пропускаем график важности признаков.")

In [19]:

# Пример: оцениваем на файле, который не использовали в обучении (например, последнем)
new_files = [almazov_data + '303.edf']

test_file = new_files[0]
evaluate_on_file(test_file, model, scaler, channels)

: 

In [ ]:
import joblib
joblib.dump(model, 'incremental_sleep_stage_model.pkl')
joblib.dump(scaler, 'scaler.pkl')